# Fase 17C — Smoke tests Baseline dos três datasets

Executa uma única rodada federada Baseline com cinco clientes para PhysioNet Challenge 2012, Dahl Rats e CheXchoNet. Congela pré-processamento e estado inicial compartilhado. Smoke tests nunca são resultados oficiais da dissertação (`usable_in_thesis=false`).

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, shutil, time
import numpy as np
import pandas as pd
import psutil, torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

CAMPAIGN_ID='THESIS_OFFICIAL_CAMPAIGN_V2_20260901'
PROJECT=Path('/content/drive/MyDrive/Mestrado_Criptografia')
CAMPAIGN=PROJECT/'OFFICIAL_CAMPAIGN_V2'/CAMPAIGN_ID
FREEZE=CAMPAIGN/'01_DATASET_FREEZE'; CONTROL=CAMPAIGN/'00_CAMPAIGN_CONTROL'; SMOKE=CAMPAIGN/'03_SMOKE_TESTS'
SEED=42; CLIENTS=5; ROUNDS=1; LOCAL_EPOCHS=3; BATCH_SIZE=64; LEARNING_RATE=0.1; THRESHOLD=0.5
torch.set_num_threads(min(2,os.cpu_count() or 1))
np.random.seed(SEED); torch.manual_seed(SEED)
gate=json.loads((CONTROL/'PHASE17B1_MASTER_GATE.json').read_text())
assert gate['training_authorized'] is True, 'Fase 17B.1 não autorizou treinamento.'
print('Baseline smoke tests autorizados para:',list(gate['dataset_approvals']))

In [ ]:
def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

def load_split(path):
    z=np.load(path,allow_pickle=True); out={}
    for k in z.files:
        lk=k.lower()
        if 'train' in lk and 'client' not in lk: out['train']=np.asarray(z[k]).reshape(-1).astype(int)
        elif ('val' in lk or 'valid' in lk) and 'client' not in lk: out['validation']=np.asarray(z[k]).reshape(-1).astype(int)
        elif 'test' in lk and 'client' not in lk: out['test']=np.asarray(z[k]).reshape(-1).astype(int)
    return out

def load_clients(path):
    z=np.load(path,allow_pickle=True)
    return [np.asarray(z[f'client_{i}']).reshape(-1).astype(int) for i in range(CLIENTS)]

def fit_transform_train_only(X,train_idx):
    X=np.asarray(X,dtype=np.float64); train=X[train_idx]
    with np.errstate(all='ignore'): median=np.nanmedian(train,axis=0)
    median=np.where(np.isfinite(median),median,0.0)
    filled=np.where(np.isfinite(X),X,median)
    mean=filled[train_idx].mean(axis=0); std=filled[train_idx].std(axis=0)
    std=np.where(np.isfinite(std)&(std>1e-12),std,1.0)
    Z=((filled-mean)/std).astype(np.float32)
    assert np.isfinite(Z).all()
    return Z,median.astype(np.float64),mean.astype(np.float64),std.astype(np.float64)

def metrics(y,prob):
    pred=(prob>=THRESHOLD).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'accuracy':float(accuracy_score(y,pred)),'precision':float(precision_score(y,pred,zero_division=0)),'recall':float(recall_score(y,pred,zero_division=0)),'f1':float(f1_score(y,pred,zero_division=0)),'auroc':float(roc_auc_score(y,prob)) if len(np.unique(y))==2 else None,'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}

def init_state(n_features):
    torch.manual_seed(SEED); model=torch.nn.Linear(n_features,1)
    return model.weight.detach().numpy().reshape(-1).astype(np.float32),model.bias.detach().numpy().reshape(-1).astype(np.float32)

def train_client(X,y,idx,w0,b0,client_id):
    model=torch.nn.Linear(X.shape[1],1); model.weight.data.copy_(torch.from_numpy(w0.reshape(1,-1))); model.bias.data.copy_(torch.from_numpy(b0))
    opt=torch.optim.SGD(model.parameters(),lr=LEARNING_RATE); loss_fn=torch.nn.BCEWithLogitsLoss(); rng=np.random.default_rng(SEED+client_id)
    start=time.perf_counter(); cpu0=time.process_time(); peak=psutil.Process().memory_info().rss
    for epoch in range(LOCAL_EPOCHS):
        order=np.asarray(idx).copy(); rng.shuffle(order)
        for s in range(0,len(order),BATCH_SIZE):
            batch=order[s:s+BATCH_SIZE]; xb=torch.from_numpy(X[batch]); yb=torch.from_numpy(y[batch].astype(np.float32)).reshape(-1,1)
            opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); opt.step(); peak=max(peak,psutil.Process().memory_info().rss)
    return model.weight.detach().numpy().reshape(-1).astype(np.float32),model.bias.detach().numpy().reshape(-1).astype(np.float32),{'client_id':client_id,'samples':len(idx),'wall_seconds':time.perf_counter()-start,'cpu_seconds':time.process_time()-cpu0,'peak_rss_bytes':int(peak)}

def predict(X,w,b):
    logits=X@w+float(b[0]); return 1/(1+np.exp(-np.clip(logits,-40,40)))

In [ ]:
DAHL_SIGNAL_FEATURES=['mean','std','min','max','median','q05','q25','q75','q95','rms','range','mean_abs_diff']

def prepare_dataset(dataset):
    root=FREEZE/dataset/'SCIENTIFIC_FREEZE'
    if dataset=='PHYSIONET_CHALLENGE_2012':
        df=pd.read_csv(root/'physionet_challenge_2012_features.csv'); cols=[c for c in df.columns if c not in {'RecordID','target'}]
        X=df[cols].apply(pd.to_numeric,errors='coerce').to_numpy(); y=pd.to_numeric(df['target']).to_numpy(dtype=int); split=load_split(root/'official_split_seed42.npz')
    elif dataset=='DAHL_RATS':
        df=pd.read_csv(root/'dahl_derived_features.csv'); missing=set(DAHL_SIGNAL_FEATURES)-set(df.columns); assert not missing,f'Features Dahl ausentes: {missing}'
        cols=DAHL_SIGNAL_FEATURES; X=df[cols].apply(pd.to_numeric,errors='coerce').to_numpy(); y=pd.to_numeric(df['target']).to_numpy(dtype=int); split=load_split(root/'official_split_seed42.npz')
    else:
        z=np.load(root/'chexchonet_embeddings.npz',allow_pickle=True); X=np.asarray(z['embeddings']); y=np.asarray(z['targets']).astype(int); cols=[f'embedding_{i:03d}' for i in range(X.shape[1])]
        raw=np.asarray(np.load(root/'official_split_original.npy',allow_pickle=True)).reshape(-1); norm=lambda v:{'0':'train','1':'validation','2':'test','val':'validation','valid':'validation','dev':'validation'}.get(str(v).strip().lower(),str(v).strip().lower()); labels=np.array([norm(v) for v in raw]); split={s:np.where(labels==s)[0] for s in ['train','validation','test']}
    clients=load_clients(root/'official_client_partition_noniid_seed42.npz'); Z,median,mean,std=fit_transform_train_only(X,split['train'])
    return root,Z,y,split,clients,cols,median,mean,std

In [ ]:
def run_baseline_smoke(dataset):
    root,X,y,split,clients,features,median,mean,std=prepare_dataset(dataset)
    timestamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    run_id=f'RUN__{dataset}__BASELINE__SMOKE__R1__C5__S42__{timestamp}'
    run_dir=SMOKE/dataset/'BASELINE'/run_id; run_dir.mkdir(parents=True,exist_ok=False)
    status={'run_id':run_id,'campaign_id':CAMPAIGN_ID,'dataset':dataset,'scenario':'BASELINE','execution_type':'SMOKE','status':'RUNNING','usable_in_thesis':False,'started_at_utc':datetime.now(timezone.utc).isoformat()}
    (run_dir/'RUN_STATUS.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
    np.savez_compressed(root/'preprocessing_train_only.npz',feature_names=np.array(features,dtype=object),median=median,mean=mean,std=std)
    w0,b0=init_state(X.shape[1]); initial=np.concatenate([w0,b0]); np.save(root/'shared_initial_state.npy',initial)
    local=[]; client_metrics=[]; round_start=time.perf_counter(); cpu0=time.process_time(); mem0=psutil.Process().memory_info().rss
    for i,idx in enumerate(clients):
        w,b,m=train_client(X,y,idx,w0,b0,i); local.append((w,b,len(idx))); client_metrics.append(m); print(dataset,'cliente',i,'concluído:',len(idx),'amostras')
    total=sum(n for _,_,n in local); w=sum(w*n for w,_,n in local)/total; b=sum(b*n for _,b,n in local)/total
    val_prob=predict(X[split['validation']],w,b); val_metrics=metrics(y[split['validation']],val_prob)
    parameter_bytes=int((len(w)+len(b))*4); communication_bytes=int(CLIENTS*2*parameter_bytes)
    round_metrics={'round':1,'wall_seconds':time.perf_counter()-round_start,'cpu_seconds':time.process_time()-cpu0,'rss_start_bytes':int(mem0),'rss_end_bytes':int(psutil.Process().memory_info().rss),'parameter_count':int(len(w)+len(b)),'parameter_bytes':parameter_bytes,'communication_bytes':communication_bytes,'aggregation':'FedAvg_weighted_by_client_samples','validation':val_metrics,'clients':client_metrics}
    np.savez_compressed(run_dir/'final_state.npz',weight=w,bias=b); pd.DataFrame(client_metrics).to_csv(run_dir/'client_metrics.csv',index=False)
    (run_dir/'round_001_metrics.json').write_text(json.dumps(round_metrics,indent=2),encoding='utf-8')
    config={'dataset':dataset,'scenario':'BASELINE','execution_type':'SMOKE','seed':SEED,'rounds':1,'clients':CLIENTS,'local_epochs':LOCAL_EPOCHS,'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,'optimizer':'SGD','loss':'BCEWithLogitsLoss','model':'LogisticRegression','feature_count':X.shape[1],'parameter_count':len(w)+len(b),'train_count':len(split['train']),'validation_count':len(split['validation']),'test_count':len(split['test']),'test_evaluated':False,'client_partition_sha256':sha256_file(root/'official_client_partition_noniid_seed42.npz'),'initial_state_sha256':sha256_file(root/'shared_initial_state.npy'),'preprocessing_sha256':sha256_file(root/'preprocessing_train_only.npz')}
    (run_dir/'RUN_CONFIG.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    checks={'five_clients_completed':len(client_metrics)==5,'finite_state':bool(np.isfinite(w).all() and np.isfinite(b).all()),'finite_validation_metrics':all(v is None or np.isfinite(v) for v in val_metrics.values()),'weighted_fedavg':True,'test_not_used':True,'artifacts_persisted':all((run_dir/x).exists() for x in ['RUN_CONFIG.json','round_001_metrics.json','final_state.npz'])}
    approved=all(checks.values()); status.update({'status':'COMPLETED_APPROVED' if approved else 'COMPLETED_REJECTED','completed_at_utc':datetime.now(timezone.utc).isoformat(),'smoke_approved':approved,'checks':checks}); (run_dir/'RUN_STATUS.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
    return {'dataset':dataset,'run_id':run_id,'run_dir':str(run_dir),'approved':approved,'validation':val_metrics,'parameter_count':int(len(w)+len(b)),'wall_seconds':round_metrics['wall_seconds'],'communication_bytes':communication_bytes,'checks':checks}

results=[]
for dataset in ['PHYSIONET_CHALLENGE_2012','DAHL_RATS','CHEXCHONET']:
    results.append(run_baseline_smoke(dataset))
print(json.dumps(results,indent=2,ensure_ascii=False))

In [ ]:
all_ok=all(r['approved'] for r in results)
master={'phase':'17C','campaign_id':CAMPAIGN_ID,'completed_at_utc':datetime.now(timezone.utc).isoformat(),'baseline_smoke_approvals':{r['dataset']:r['approved'] for r in results},'all_baseline_smokes_approved':all_ok,'official_training_authorized':False,'next_authorized_step':'FASE_17D_CKKS_SMOKE_TESTS' if all_ok else 'REPAIR_BASELINE_SMOKE_FAILURES','note':'Smoke tests não podem ser usados na dissertação. A campanha oficial de 30 rodadas permanece bloqueada até Baseline, CKKS e Hybrid passarem em smoke tests.','results':results}
(CONTROL/'PHASE17C_MASTER_GATE.json').write_text(json.dumps(master,indent=2,ensure_ascii=False),encoding='utf-8')
status=json.loads((CONTROL/'CAMPAIGN_STATUS.json').read_text()); status.update({'status':'PHASE17C_COMPLETED' if all_ok else 'PHASE17C_BLOCKED','phase17c_baseline_smokes_approved':all_ok,'next_authorized_step':master['next_authorized_step']}); (CONTROL/'CAMPAIGN_STATUS.json').write_text(json.dumps(status,indent=2,ensure_ascii=False),encoding='utf-8')
print('='*100); print(json.dumps(master,indent=2,ensure_ascii=False)); print('='*100)

In [ ]:
export=Path('/content/PHASE17C_EVIDENCE'); shutil.rmtree(export,ignore_errors=True); export.mkdir()
for n in ['CAMPAIGN_STATUS.json','PHASE17B1_MASTER_GATE.json','PHASE17C_MASTER_GATE.json']: shutil.copy2(CONTROL/n,export/n)
for r in results:
    dst=export/r['dataset']; shutil.copytree(Path(r['run_dir']),dst,ignore=shutil.ignore_patterns('final_state.npz'))
zip_path=shutil.make_archive('/content/PHASE17C_EVIDENCE','zip','/content','PHASE17C_EVIDENCE'); permanent=CONTROL/'EXPORTS'/'PHASE17C_EVIDENCE.zip'; permanent.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(zip_path,permanent)
print('Salvo em:',permanent); files.download(str(permanent))